In [1]:
# ------ Single ticker test -------

In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from fineas.ingest.yfinance_ohlcv import YFIngestParams, fetch_yf_ohlcv, canonicalize_yf_df, write_month_partitions
from fineas.io.qc import qc_raw_ohlcv_partition

In [3]:
params = YFIngestParams(interval="1d", start="2022-01-01", end="2024-01-01", auto_adjust=False, actions=True)

raw = fetch_yf_ohlcv("AAPL", params)
type(raw.columns), getattr(raw.columns, "names", None), raw.shape

(pandas.MultiIndex, FrozenList(['Price', 'Ticker']), (501, 8))

In [4]:
canon = canonicalize_yf_df(raw, "AAPL")
canon.columns, canon.shape, canon.head()

(Index(['ts', 'symbol', 'open', 'high', 'low', 'close', 'adj_close', 'volume',
        'dividends', 'stock_splits'],
       dtype='str', name='Price'),
 (501, 10),
 Price                        ts symbol        open        high         low  \
 0     2022-01-03 00:00:00+00:00   AAPL  177.830002  182.880005  177.710007   
 1     2022-01-04 00:00:00+00:00   AAPL  182.630005  182.940002  179.119995   
 2     2022-01-05 00:00:00+00:00   AAPL  179.610001  180.169998  174.639999   
 3     2022-01-06 00:00:00+00:00   AAPL  172.699997  175.300003  171.639999   
 4     2022-01-07 00:00:00+00:00   AAPL  172.889999  174.139999  171.029999   
 
 Price       close   adj_close     volume  dividends  stock_splits  
 0      182.009995  178.103638  104487900        0.0           0.0  
 1      179.699997  175.843216   99310400        0.0           0.0  
 2      174.919998  171.165833   94537600        0.0           0.0  
 3      172.000000  168.308517   96904000        0.0           0.0  
 4      172.169

In [5]:
m = canon[(canon["ts"].dt.year == 2022) & (canon["ts"].dt.month == 1)].copy()
qc = qc_raw_ohlcv_partition(m, name="qc:AAPL/2022-01")
qc.to_dict()

{'ok': True,
 'hard_fails': [],
 'warnings': [],
 'stats': {'name': 'qc:AAPL/2022-01',
  'ts_min': '2022-01-03 00:00:00+00:00',
  'ts_max': '2022-01-31 00:00:00+00:00',
  'key_name': 'qc:AAPL/2022-01/key',
  'key_rows': 20,
  'key_key_cols': ['symbol', 'ts'],
  'key_dup_keys_n': 0,
  'time_name': 'qc:AAPL/2022-01/time',
  'time_monotonic_bad_symbols': [],
  'time_max_gap_days': 4.0,
  'time_gap_warn_symbols': [],
  'ohlcv_name': 'qc:AAPL/2022-01/ohlcv',
  'ohlcv_rows': 20,
  'ohlcv_ohlc_violation_rows': 0,
  'ohlcv_volume_negative_rows': 0,
  'ohlcv_nan_counts': {'open': 0,
   'high': 0,
   'low': 0,
   'close': 0,
   'volume': 0}}}

In [6]:
rep = write_month_partitions(canon, interval="1d", repo_root=ROOT)
rep["status"], rep["parts"][:2]

('ok',
 [{'year': 2022,
   'month': 1,
   'rows': 20,
   'path': 'C:\\Users\\quantbase\\Desktop\\fineas\\data\\raw\\yfinance\\ohlcv\\interval=1d\\symbol=AAPL\\year=2022\\month=01\\part-2022-01.parquet',
   'qc': {'ok': True,
    'hard_fails': [],
    'warnings': [],
    'stats': {'name': 'raw_yf_ohlcv:AAPL/1d/2022-01',
     'ts_min': '2022-01-03 00:00:00+00:00',
     'ts_max': '2022-01-31 00:00:00+00:00',
     'key_name': 'raw_yf_ohlcv:AAPL/1d/2022-01/key',
     'key_rows': 20,
     'key_key_cols': ['symbol', 'ts'],
     'key_dup_keys_n': 0,
     'time_name': 'raw_yf_ohlcv:AAPL/1d/2022-01/time',
     'time_monotonic_bad_symbols': [],
     'time_max_gap_days': 4.0,
     'time_gap_warn_symbols': [],
     'ohlcv_name': 'raw_yf_ohlcv:AAPL/1d/2022-01/ohlcv',
     'ohlcv_rows': 20,
     'ohlcv_ohlc_violation_rows': 0,
     'ohlcv_volume_negative_rows': 0,
     'ohlcv_nan_counts': {'open': 0,
      'high': 0,
      'low': 0,
      'close': 0,
      'volume': 0}}},
   'status': 'ok'},
  {'year

In [7]:
#------- Basket ------

In [16]:
import subprocess
import sys
from pathlib import Path

# repo root = one level above notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

cmd = [sys.executable, str(ROOT / "scripts" / "01_ingest_yf.py")]
print("Running:", " ".join(cmd))

r = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
print("returncode:", r.returncode)
print("stdout:\n", r.stdout[:2000])
print("stderr:\n", r.stderr[:2000])
assert r.returncode == 0, "ingestion script returned non-zero (see stderr/stdout)"

Running: c:\Users\quantbase\.conda\envs\fineas\python.exe C:\Users\quantbase\Desktop\fineas\scripts\01_ingest_yf.py
returncode: 0
stdout:
 
stderr:
 


In [17]:
import json

runs_dir = ROOT / "data" / "meta" / "runs"
latest = sorted([p for p in runs_dir.glob("run=*") if p.is_dir()])[-1]
report_path = latest / "ingest_report.json"
print("report:", report_path)

payload = json.loads(report_path.read_text(encoding="utf-8"))
payload["results"]["summary"]

report: C:\Users\quantbase\Desktop\fineas\data\meta\runs\run=20260304T102000Z\ingest_report.json


{'provider': 'yfinance',
 'interval': '1d',
 'start': '2022-01-01',
 'end': '2024-01-01',
 'symbols_total': 4,
 'ok': 4,
 'failed': 0,
 'empty': 0}

In [18]:
parts = []
for it in payload["results"]["items"]:
    pr = it.get("part_report", {})
    for p in pr.get("parts", []):
        if p.get("status") == "ok":
            parts.append(p["path"])

len(parts), parts[:3]

(96,
 ['C:\\Users\\quantbase\\Desktop\\fineas\\data\\raw\\yfinance\\ohlcv\\interval=1d\\symbol=AAPL\\year=2022\\month=01\\part-2022-01.parquet',
  'C:\\Users\\quantbase\\Desktop\\fineas\\data\\raw\\yfinance\\ohlcv\\interval=1d\\symbol=AAPL\\year=2022\\month=02\\part-2022-02.parquet',
  'C:\\Users\\quantbase\\Desktop\\fineas\\data\\raw\\yfinance\\ohlcv\\interval=1d\\symbol=AAPL\\year=2022\\month=03\\part-2022-03.parquet'])

In [19]:
import pandas as pd
from fineas.io.qc import qc_raw_ohlcv_partition

sample_path = Path(parts[0])
df = pd.read_parquet(sample_path)
df.head(), df.dtypes

(                         ts symbol        open        high         low  \
 0 2022-01-03 00:00:00+00:00   AAPL  177.830002  182.880005  177.710007   
 1 2022-01-04 00:00:00+00:00   AAPL  182.630005  182.940002  179.119995   
 2 2022-01-05 00:00:00+00:00   AAPL  179.610001  180.169998  174.639999   
 3 2022-01-06 00:00:00+00:00   AAPL  172.699997  175.300003  171.639999   
 4 2022-01-07 00:00:00+00:00   AAPL  172.889999  174.139999  171.029999   
 
         close   adj_close     volume  dividends  stock_splits  
 0  182.009995  178.103653  104487900        0.0           0.0  
 1  179.699997  175.843246   99310400        0.0           0.0  
 2  174.919998  171.165817   94537600        0.0           0.0  
 3  172.000000  168.308533   96904000        0.0           0.0  
 4  172.169998  168.474854   86709100        0.0           0.0  ,
 ts              datetime64[ms, UTC]
 symbol                          str
 open                        float64
 high                        float64
 low     

In [20]:
qc = qc_raw_ohlcv_partition(df, name=f"recheck:{sample_path.name}")
qc.to_dict()

{'ok': True,
 'hard_fails': [],
 'warnings': [],
 'stats': {'name': 'recheck:part-2022-01.parquet',
  'ts_min': '2022-01-03 00:00:00+00:00',
  'ts_max': '2022-01-31 00:00:00+00:00',
  'key_name': 'recheck:part-2022-01.parquet/key',
  'key_rows': 20,
  'key_key_cols': ['symbol', 'ts'],
  'key_dup_keys_n': 0,
  'time_name': 'recheck:part-2022-01.parquet/time',
  'time_monotonic_bad_symbols': [],
  'time_max_gap_days': 4.0,
  'time_gap_warn_symbols': [],
  'ohlcv_name': 'recheck:part-2022-01.parquet/ohlcv',
  'ohlcv_rows': 20,
  'ohlcv_ohlc_violation_rows': 0,
  'ohlcv_volume_negative_rows': 0,
  'ohlcv_nan_counts': {'open': 0,
   'high': 0,
   'low': 0,
   'close': 0,
   'volume': 0}}}

In [21]:
qc_failed = []
qc_warn = []

for it in payload["results"]["items"]:
    pr = it.get("part_report", {})
    for p in pr.get("parts", []):
        q = p.get("qc", {})
        if p.get("status") == "qc_failed":
            qc_failed.append((p["path"], q.get("hard_fails", [])))
        if q.get("warnings"):
            qc_warn.append((p["path"], q.get("warnings")))

qc_failed[:3], len(qc_failed), len(qc_warn)

([], 0, 0)

In [22]:
import pandas as pd
import numpy as np

t = df.sort_values("ts")["ts"]
diffs_days = t.diff().dt.total_seconds().dropna() / 86400
diffs_days.max(), diffs_days.head(10).to_numpy()

(np.float64(4.0), array([1., 1., 1., 1., 3., 1., 1., 1., 1., 4.]))

In [23]:
qc.stats["time_max_gap_days"]

4.0